# Rolling realized volatility — any ticker

Computes and plots rolling 1-year (252-day) and 3-year (756-day) annualized realized volatility.
Change `TICKER` (and optionally `LABEL`) in the **Parameters** cell below.

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
TICKER = "SPY"          # Any Yahoo Finance ticker: SPY, GLD, USO, ^GSPC, EURUSD=X …
LABEL  = ""             # Display name — leave empty to use TICKER
START  = "2000-01-01"   # Long start so 3Y rolling has room to warm up

WIN_1Y = 252
WIN_3Y = 756

_label = LABEL or TICKER

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

from alpha_lab.data.loaders.yfinance import load_prices

## 1. Price history

In [ ]:
prices = load_prices(TICKER, start=START)
px = prices.squeeze()   # single-column → Series

fig, ax = plt.subplots(figsize=(12, 3.5), constrained_layout=True)
ax.plot(px.index, px.values, color="#2c7bb6", linewidth=1.2, label=_label)
ax.fill_between(px.index, px.values, px.min(), alpha=0.08, color="#2c7bb6")
ax.set_ylabel("Price")
ax.set_title(f"{_label} — price history", fontsize=12, pad=8)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 2. Rolling realized volatility

Annualized realized vol = rolling std of log returns × √252.

In [ ]:
log_ret = np.log(px / px.shift(1)).dropna()

vol_1y = log_ret.rolling(WIN_1Y).std() * np.sqrt(252)
vol_3y = log_ret.rolling(WIN_3Y).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)

ax.plot(vol_1y.index, vol_1y.values, color="#e74c3c", linewidth=1.2,
        label="1Y rolling vol")
ax.plot(vol_3y.index, vol_3y.values, color="#2c3e50", linewidth=1.5,
        linestyle="--", label="3Y rolling vol")

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("Annualized volatility")
ax.set_title(f"{_label} — rolling realized volatility", fontsize=12, pad=8)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 3. Current levels vs. history

In [ ]:
summary = pd.DataFrame({"1Y vol": vol_1y, "3Y vol": vol_3y}).dropna()

current = summary.iloc[[-1]].rename(index=lambda _: "Current")
stats   = summary.describe().loc[["mean", "50%", "min", "max"]].rename(index={"50%": "median"})

(
    pd.concat([current, stats])
    .style
    .format("{:.1%}")
    .set_caption(f"{_label} realized vol — current vs. history")
)

## 4. Vol regime bands

Shade the 1Y vol series by quartile bucket to quickly see how the current environment compares to history.

In [ ]:
# Quartile thresholds from full history
q25, q50, q75 = vol_1y.quantile([0.25, 0.50, 0.75])

fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)

# Shaded regime bands
ax.axhspan(0,    q25, alpha=0.06, color="#2ecc71", label="Low (<Q1)")
ax.axhspan(q25,  q50, alpha=0.06, color="#f1c40f", label="Below median")
ax.axhspan(q50,  q75, alpha=0.06, color="#e67e22", label="Above median")
ax.axhspan(q75,  vol_1y.max() * 1.1, alpha=0.06, color="#e74c3c", label="High (>Q3)")

ax.plot(vol_1y.index, vol_1y.values, color="#2c3e50", linewidth=1.2, zorder=3)

# Mark current level
latest_val = vol_1y.dropna().iloc[-1]
latest_dt  = vol_1y.dropna().index[-1]
ax.axhline(latest_val, color="#e74c3c", linewidth=0.8, linestyle=":", alpha=0.7)
ax.annotate(
    f"Now: {latest_val:.1%}",
    xy=(latest_dt, latest_val),
    xytext=(-80, 8), textcoords="offset points",
    fontsize=9, color="#e74c3c",
    arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=0.8),
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("1Y realized vol")
ax.set_title(f"{_label} — vol regime bands (1Y rolling)", fontsize=12, pad=8)
ax.legend(loc="upper left", fontsize=8, framealpha=0.85)
ax.set_ylim(bottom=0)
ax.spines[["top", "right"]].set_visible(False)
plt.show()